In [1]:
import os
import zipfile
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


unzip 

In [4]:
zip_path = "/content/drive/MyDrive/scenario23_dev_w_resources.zip"
print("Exists:", os.path.exists(zip_path))

Exists: True


In [6]:
extract_dir = "/content/dataset"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

print("Unzipped to:", extract_dir)

Unzipped to: /content/dataset


In [7]:
for root, dirs, files in os.walk(extract_dir):
    if "scenario23" in root.lower():
        print(root)

/content/dataset/scenario23_dev
/content/dataset/scenario23_dev/resources
/content/dataset/scenario23_dev/resources/bbox_labels_final
/content/dataset/scenario23_dev/unit1
/content/dataset/scenario23_dev/unit1/camera_data
/content/dataset/scenario23_dev/unit1/GPS_data
/content/dataset/scenario23_dev/unit1/mmWave_data
/content/dataset/scenario23_dev/unit2
/content/dataset/scenario23_dev/unit2/altitude
/content/dataset/scenario23_dev/unit2/height
/content/dataset/scenario23_dev/unit2/distance
/content/dataset/scenario23_dev/unit2/y_speed
/content/dataset/scenario23_dev/unit2/z_speed
/content/dataset/scenario23_dev/unit2/roll
/content/dataset/scenario23_dev/unit2/GPS_data
/content/dataset/scenario23_dev/unit2/speed
/content/dataset/scenario23_dev/unit2/x_speed
/content/dataset/scenario23_dev/unit2/pitch


In [8]:
! find /content/dataset -maxdepth 4 | head -200

/content/dataset
/content/dataset/scenario23_dev
/content/dataset/scenario23_dev/resources
/content/dataset/scenario23_dev/resources/bbox_labels_final
/content/dataset/scenario23_dev/resources/bbox_labels_final/image_BS1_10012_17_56_04.txt
/content/dataset/scenario23_dev/resources/bbox_labels_final/image_BS1_10013_17_56_04.txt
/content/dataset/scenario23_dev/resources/bbox_labels_final/image_BS1_10014_17_56_05.txt
/content/dataset/scenario23_dev/resources/bbox_labels_final/image_BS1_10015_17_56_05.txt
/content/dataset/scenario23_dev/resources/bbox_labels_final/image_BS1_10016_17_56_05.txt
/content/dataset/scenario23_dev/resources/bbox_labels_final/image_BS1_10017_17_56_05.txt
/content/dataset/scenario23_dev/resources/bbox_labels_final/image_BS1_10018_17_56_05.txt
/content/dataset/scenario23_dev/resources/bbox_labels_final/image_BS1_10019_17_56_05.txt
/content/dataset/scenario23_dev/resources/bbox_labels_final/image_BS1_1001_17_01_10.txt
/content/dataset/scenario23_dev/resources/bbox_la

In [9]:
DATA_ROOT = "/content/dataset/scenario23_dev"   # update if needed
print("Exists:", os.path.exists(DATA_ROOT))

Exists: True


In [10]:
csv_files = []
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        if f.lower().endswith(".csv"):
            csv_files.append(os.path.join(root, f))

print("Total CSV files:", len(csv_files))
for f in csv_files[:50]:
    print(f)

Total CSV files: 1
/content/dataset/scenario23_dev/scenario23.csv


In [11]:
image_exts = (".jpg", ".jpeg", ".png", ".bmp")
img_examples = []

for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        if f.lower().endswith(image_exts):
            img_examples.append(os.path.join(root, f))

print("Total sample image hits:", len(img_examples))
for x in img_examples[:20]:
    print(x)

Total sample image hits: 11387
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_604_17_00_05.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_1410_17_02_16.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_5394_17_14_48.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_6760_17_46_03.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_23_16_58_45.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_9695_17_55_15.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_9037_17_53_37.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_41_16_58_47.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_11563_18_00_53.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_5201_17_14_16.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3800_17_09_05.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2790_17_06_07.jpg
/content/dataset/scenario23_dev/unit1

In [12]:
for path in csv_files[:10]:
    print("="*100)
    print(path)
    try:
        df_tmp = pd.read_csv(path)
        print(df_tmp.head())
        print(df_tmp.columns.tolist())
        print(df_tmp.shape)
    except Exception as e:
        print("Could not read:", e)

/content/dataset/scenario23_dev/scenario23.csv
   index                                     unit1_rgb  \
0      1  ./unit1/camera_data/image_BS1_1_16_58_42.jpg   
1      2  ./unit1/camera_data/image_BS1_2_16_58_42.jpg   
2      3  ./unit1/camera_data/image_BS1_3_16_58_42.jpg   
3      4  ./unit1/camera_data/image_BS1_4_16_58_42.jpg   
4      5  ./unit1/camera_data/image_BS1_5_16_58_42.jpg   

                          unit1_pwr_60ghz                          unit1_loc  \
0  ./unit1/mmWave_data/mmWave_power_1.txt  ./unit1/GPS_data/gps_location.txt   
1  ./unit1/mmWave_data/mmWave_power_2.txt  ./unit1/GPS_data/gps_location.txt   
2  ./unit1/mmWave_data/mmWave_power_3.txt  ./unit1/GPS_data/gps_location.txt   
3  ./unit1/mmWave_data/mmWave_power_4.txt  ./unit1/GPS_data/gps_location.txt   
4  ./unit1/mmWave_data/mmWave_power_5.txt  ./unit1/GPS_data/gps_location.txt   

                             unit2_loc                unit2_speed  \
0  ./unit2/GPS_data/gps_location_1.txt  ./unit2/speed/

In [13]:
candidate_tables = []

for path in csv_files:
    try:
        df_tmp = pd.read_csv(path, nrows=5)
        cols = [c.lower() for c in df_tmp.columns]
        if any("beam" in c for c in cols) or any("rgb" in c for c in cols):
            candidate_tables.append(path)
    except:
        pass

print("Candidate metadata tables:")
for p in candidate_tables:
    print(p)

Candidate metadata tables:
/content/dataset/scenario23_dev/scenario23.csv


inspect important columns 

# for loop diye column dekhe column mapping korbo 

In [14]:
meta_csv = candidate_tables[0]   # update manually if needed
df = pd.read_csv(meta_csv)
print(df.shape)
print(df.head())
print(df.columns.tolist())

(11387, 17)
   index                                     unit1_rgb  \
0      1  ./unit1/camera_data/image_BS1_1_16_58_42.jpg   
1      2  ./unit1/camera_data/image_BS1_2_16_58_42.jpg   
2      3  ./unit1/camera_data/image_BS1_3_16_58_42.jpg   
3      4  ./unit1/camera_data/image_BS1_4_16_58_42.jpg   
4      5  ./unit1/camera_data/image_BS1_5_16_58_42.jpg   

                          unit1_pwr_60ghz                          unit1_loc  \
0  ./unit1/mmWave_data/mmWave_power_1.txt  ./unit1/GPS_data/gps_location.txt   
1  ./unit1/mmWave_data/mmWave_power_2.txt  ./unit1/GPS_data/gps_location.txt   
2  ./unit1/mmWave_data/mmWave_power_3.txt  ./unit1/GPS_data/gps_location.txt   
3  ./unit1/mmWave_data/mmWave_power_4.txt  ./unit1/GPS_data/gps_location.txt   
4  ./unit1/mmWave_data/mmWave_power_5.txt  ./unit1/GPS_data/gps_location.txt   

                             unit2_loc                unit2_speed  \
0  ./unit2/GPS_data/gps_location_1.txt  ./unit2/speed/speed_1.txt   
1  ./unit2/GPS_data/

In [15]:
for col in df.columns:
    print(col)

index
unit1_rgb
unit1_pwr_60ghz
unit1_loc
unit2_loc
unit2_speed
unit2_altitude
unit2_distance
unit2_height
unit2_x-speed
unit2_y-speed
unit2_z-speed
unit2_pitch
unit2_roll
seq_index
time_stamp[UTC]
unit1_beam_index


column mapping 

In [16]:
COLUMN_MAP = {
    "image_col": None,
    "lat_col": None,
    "lon_col": None,
    "height_col": None,
    "distance_col": None,
    "beam32_col": None,
    "beam64_col": None,
}

In [17]:
cols = df.columns.tolist()
lower_map = {c.lower(): c for c in cols}

print("Possible image columns:")
for c in cols:
    if "rgb" in c.lower() or "image" in c.lower() or "camera" in c.lower():
        print(c)

print("\nPossible latitude/longitude columns:")
for c in cols:
    if "lat" in c.lower() or "lon" in c.lower() or "long" in c.lower() or "gps" in c.lower():
        print(c)

print("\nPossible height/distance columns:")
for c in cols:
    if "height" in c.lower() or "dist" in c.lower():
        print(c)

print("\nPossible beam columns:")
for c in cols:
    if "beam" in c.lower():
        print(c)

Possible image columns:
unit1_rgb

Possible latitude/longitude columns:

Possible height/distance columns:
unit2_distance
unit2_height

Possible beam columns:
unit1_beam_index


In [18]:
COLUMN_MAP = {
    "image_col": "unit1_rgb",
    "lat_col": "unit2_loc_0",      # update
    "lon_col": "unit2_loc_1",      # update
    "height_col": "unit2_height",  # update
    "distance_col": "unit1_unit2_distance",  # update
    "beam32_col": "unit1_beam_32",
    "beam64_col": "unit1_beam_64",
}

COLUMN_MAP

{'image_col': 'unit1_rgb',
 'lat_col': 'unit2_loc_0',
 'lon_col': 'unit2_loc_1',
 'height_col': 'unit2_height',
 'distance_col': 'unit1_unit2_distance',
 'beam32_col': 'unit1_beam_32',
 'beam64_col': 'unit1_beam_64'}

In [19]:
for key, col in COLUMN_MAP.items():
    if col is not None and col in df.columns:
        print(f"{key} -> {col} | nulls = {df[col].isna().sum()}")

image_col -> unit1_rgb | nulls = 0
height_col -> unit2_height | nulls = 0


In [20]:
def make_abs_path(p):
    if pd.isna(p):
        return None
    p = str(p)
    if os.path.isabs(p):
        return p
    return os.path.join("/content", p.lstrip("/"))

df["image_path"] = df[COLUMN_MAP["image_col"]].apply(make_abs_path)

print(df["image_path"].head())
print("Existing images:", df["image_path"].apply(lambda x: os.path.exists(x) if isinstance(x, str) else False).sum())

0    /content/./unit1/camera_data/image_BS1_1_16_58...
1    /content/./unit1/camera_data/image_BS1_2_16_58...
2    /content/./unit1/camera_data/image_BS1_3_16_58...
3    /content/./unit1/camera_data/image_BS1_4_16_58...
4    /content/./unit1/camera_data/image_BS1_5_16_58...
Name: image_path, dtype: object
Existing images: 0


In [22]:
if COLUMN_MAP["beam32_col"] in df.columns:
    df["label"] = pd.to_numeric(df[COLUMN_MAP["beam32_col"]], errors="coerce")
else:
    raise ValueError("beam32_col not found. Need raw power-vector pipeline separately.")

print(df["label"].head())
print("Num classes:", df["label"].nunique())
print("Min label:", df["label"].min(), "Max label:", df["label"].max())

ValueError: beam32_col not found. Need raw power-vector pipeline separately.